# NYC 311 Noise Complaints — Explainer Notebook

This notebook is the behind-the-scenes companion to our project website. It
documents how we loaded, cleaned, and analyzed NYC 311 noise complaint data,
and explains the choices behind the visualizations shown on the site.

**Dataset:** NYC 311 Service Requests, filtered to noise-related complaints and dates
(2018–2025), downloaded from NYC Open Data.

## Motivation

Our project is built on NYC 311 Service Requests, an open dataset published by NYC Open Data that records every non-emergency complaint and service request made by residents of New York City. Whenever someone dials 311, fills out a form on the city's website, or uses the official app to report a problem, the interaction is logged here. The dataset covers everything from potholes and broken streetlights to rats, illegal parking, graffiti, and noise. Each record contains a timestamp, the type and description of the complaint, the borough and (usually) the latitude/longitude where it occurred, and which agency was responsible for handling it. NYC Open Data splits it across two portals: the 311 Service Requests from 2010 to 2019 archive, and the 311 Service Requests from 2020 to Present live feed. For our project, we narrow the scope to noise complaints between 2018 and 2025, which still leaves us with millions of records.

We chose this dataset because it offers a window into the lives of New Yorkers. Each record is a small act of civic engagement. It tells us that someone, somewhere, was annoyed or worried enough to pick up the phone. Aggregated across millions of calls, those individual frustrations paint an honest portrait of urban life, where the city is loud, people bother each other, and people want to feel heard. The dataset is also large, rich, and geo-temporally tagged, making it ideal for the analyses and visualizations we have learned in this course. Furthermore, NYC is a city most readers can picture even if they have never been. We chose noise specifically because it is universal and intuitive. Everyone has been kept awake by a neighbor, a party, or a construction crew. And at the scale of an entire metropolis over multiple years, noise complaints start revealing certain patterns. The rhythm of weekdays versus weekends, the impact of the COVID-19 lockdowns, and the difference between a residential block in Staten Island and a lively nightlife neighbourhood in Manhattan.

Our goal is for a curious, non-technical reader to come away with a clear understanding of how, when, and where New York City complains about noise, and what those patterns reveal about the city itself. We want the website to feel less like a report and more like a guided tour: starting from the big picture, zooming into the patterns over time and space, and ending with the human-scale takeaways.

## Data cleaning and preprocessing

**Pipeline:** raw CSVs in `../data/raw/` → cleaned file written to
`../data/processed/311_noise_cleaned.csv` → used by the visualization team
for the website.

Standard data science stack (pandas, matplotlib, seaborn). Warnings
suppressed globally to keep output clean. Paths are defined as constants
so anyone can edit them in one place if their local layout differs.

In [ ]:
import os
import warnings

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Suppress warnings globally
warnings.filterwarnings("ignore")

# Paths — edit if your local layout differs
DATA_DIR = "../data/raw"    # the two raw NYC 311 CSVs go here
OUT_DIR  = "../data/processed"   # cleaned output lands here
os.makedirs(OUT_DIR, exist_ok=True)

### Load and merge datasets

NYC Open Data publishes 311 service requests as two separate datasets:

- [311 Service Requests 2010–2019](https://data.cityofnewyork.us/Social-Services/311-Service-Requests-from-2010-to-2019/76ig-c548/about_data) 
- [311 Service Requests 2020–Present](https://data.cityofnewyork.us/Social-Services/311-Service-Requests-from-2020-to-Present/erm2-nwe9/about_data)

Two filters
were applied at export time:

- **Complaint Type** filtered to noise-related categories only
- **Created Date** between 2018-01-01 and 2025-12-31

After loading both CSV files, we merged them into a single dataframe. 

To reproduce, apply the same filters at both datasets, download as CSV, and save as:
- "311_Service_Requests_from_2018_to_2019_20260413.csv"
- "311_Service_Requests_from_2020_to_2025_20260413.csv"

finally, drop the two files into `../data/raw/`.

In [ ]:
# Load both CSVs
file_path1 = os.path.join(DATA_DIR, "311_Service_Requests_from_2018_to_2019_20260413.csv")
file_path2 = os.path.join(DATA_DIR, "311_Service_Requests_from_2020_to_2025_20260413.csv")

df1 = pd.read_csv(file_path1)
df2 = pd.read_csv(file_path2)

# Merge both datasets
df = pd.concat([df1, df2], ignore_index=True)

## Clean and save dataset

We reduce the raw export to the columns we'll actually use and drop rows
that can't support our analysis. Four decisions were made here:

1. **Keep 10 columns.** Timestamp, complaint type and detail, location
   type, borough, lat/long, and three street-level fields. The raw export
   has ~40 columns of administrative metadata (agency routing, closure
   timestamps, etc.) that we don't need for the analysis we are doing.
2. **Drop rows where Borough is "Unspecified" or null.** Our analysis
   groups by borough, so these rows can't contribute.
3. **Require coordinates.** Several planned visualizations are map-based,
   and complaints without latitude/longitude can't be placed.
4. **Drop duplicates.** Exact duplicates are rare in 311 data but cheap
   to remove as insurance.

We also parse `Created Date` into a proper datetime here so downstream
cells don't have to re-parse it.

The notebook assumes that the raw CSVs were exported
with the noise-type filter already applied at NYC Open Data. As a safety
net, an `assert` verifies that every complaint in the loaded data starts
with `"Noise"`. If someone re-exports the CSVs without the filter, the
notebook will fail loudly here rather than silently producing wrong
results downstream.

The cleaned file is saved to `../data/processed/311_noise_cleaned.csv`.

In [ ]:
# Keep only columns relevant to the analysis
cols_to_keep = [
    'Created Date',
    'Problem (formerly Complaint Type)',
    'Problem Detail (formerly Descriptor)',
    'Location Type',
    'Borough',
    'Latitude',
    'Longitude',
    # Optional street-level
    'Incident Address',
    'Cross Street 1',
    'Cross Street 2',
]
df = df[cols_to_keep].rename(columns={
    'Problem (formerly Complaint Type)': 'Complaint Type',
    'Problem Detail (formerly Descriptor)': 'Complaint description',
})

# Safety net: confirm the export filter was actually applied.
# If this fails, someone exported the CSVs without the "Noise" filter.
assert df['Complaint Type'].str.contains('Noise', na=False, case=False).all(), \
    "Non-noise complaints found — check the export filter at NYC Open Data"

# Drop rows we can't analyze
df = df[(df['Borough'] != 'Unspecified') & (df['Borough'].notna())]
df = df.dropna(subset=['Latitude', 'Longitude'])
df = df.drop_duplicates()

# Parse date column once. Here, downstream cells rely on this
df['Created Date'] = pd.to_datetime(df['Created Date'], errors='coerce')

# Save cleaned dataset
output_path = os.path.join(OUT_DIR, "311_noise_cleaned.csv")
df.to_csv(output_path, index=False)

We now load the cleaned dataset so the analysis below can run quickly without re-running the full cleaning pipeline.

In [ ]:
import pandas as pd
df = pd.read_csv("../data/processed/311_noise_cleaned.csv", parse_dates=["Created Date"])

## Basic Stats

After cleaning, we have ~5.5M noise complaints spanning 2018 through 2025.
Missingness is low on the fields we care about. `Location Type` is the
only column with meaningful missing data (~9%). That won't affect
borough or time-based analyses. All geographic and temporal fields
are complete.

In [ ]:
# Basic shape
print(f"Rows:     {len(df):,}")
print(f"Columns:  {len(df.columns)}")
print(f"Range:    {df['Created Date'].min().date()} to {df['Created Date'].max().date()}")
print(f"Memory:   {df.memory_usage(deep=True).sum() / 1024**2:.0f} MB")

# Column names
print("\nColumns:")
print(df.columns.to_list())

# Missing values
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_df = pd.DataFrame({
    'Missing': missing,
    'Percent': (missing / len(df) * 100).round(2)
})
print("\nMissing values by column:")
print(missing_df.to_string() if len(missing_df) else "(none)")

### Basic Stats: Exploratory Data Analysis

To better understand the underlying patterns in our dataset, we conducted an exploratory data analysis focusing on the geographic, categorical, and temporal distribution of the complaints. The visualizations below highlight the key trends that will drive the narrative of our final data story.

**Geographic Distribution (Complaints by Borough)**: The data reveals a stark contrast in complaint volumes across the city. The **Bronx registers the highest number of complaints (roughly 1.5 million)**, closely followed by Manhattan and Brooklyn. Staten Island accounts for the lowest volume by a significant margin. This suggests that complaint frequency is likely tied to population density, urban layout, and borough-specific demographics.

In [ ]:
sns.set_style('whitegrid')

# Plot 1: Borough
borough_counts = df['Borough'].value_counts()
borough_share = (borough_counts / borough_counts.sum() * 100).round(2)
borough_summary = pd.DataFrame({
    'Count': borough_counts,
    'Percent': borough_share,
})
print("Borough counts and share (%):")
print(borough_summary.to_string())

fig, ax = plt.subplots(figsize=(10, 5))
borough_counts.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Complaints by Borough', fontweight='bold', fontsize=12)
ax.set_ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

**Categorical Breakdown (Complaints by Type)**: When categorizing the data, "Residential" complaints absolutely dominate the dataset, exceeding 2.5 million records, with "Street/Sidewalk" as the second most prevalent category. However, looking at the broad categories only tells half the story.

To understand what New Yorkers are actually complaining about, we looked at the specific sub-descriptors. The data reveals that the vast majority of these issues are behavioral rather than mechanical or infrastructural. "Loud Music/Party" is the undisputed leading cause of noise complaints citywide (over 3 million records). This is followed by "Banging/Pounding" and "Loud Talking". This clear disparity indicates that the core issue we are analyzing is primarily social noise that affects residents in or immediately around their homes.

In [ ]:
# Plot 2: Complaints by type
fig, ax = plt.subplots(figsize=(10, 5))
counts = df['Complaint Type'].value_counts()
counts.index = counts.index.str.replace(r'^Noise\s*-\s*', '', regex=True)
counts.plot(kind='bar', ax=ax, color='coral')
ax.set_title('Complaints by Type', fontweight='bold', fontsize=12)
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
top_desc = df['Complaint description'].value_counts().head(10).sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
top_desc.plot(kind='barh', ax=ax, color='seagreen')
ax.set_title('Top 10 noise descriptors', fontweight='bold')
ax.set_xlabel('Complaints')
ax.set_ylabel('Descriptor')
plt.tight_layout()
plt.show()

**Long-Term Temporal Trends (Complaints Over Time)**:
Looking at the timeline from 2018 through 2025, a dramatic anomaly stands out: a massive spike in complaints occurring in mid-2020. This aligns perfectly with the onset of the COVID-19 pandemic and subsequent lockdowns, a period when more people were confined to their residences. Following this 2020 surge, the data settles into a highly pronounced, recurring seasonal pattern, with sharp peaks happening annually (likely during the warmer summer months).

In [ ]:
# Plot 3: Complaints over time
complaints_per_month = df['Created Date'].dt.to_period('M').value_counts().sort_index()
fig, ax = plt.subplots(figsize=(12, 5))
complaints_per_month.plot(kind='line', ax=ax, color='darkblue', linewidth=2)
ax.set_title('Complaints Over Time', fontweight='bold', fontsize=12)
ax.set_ylabel('Count')
ax.set_xlabel('Month')
plt.tight_layout()
plt.show()

**Short-Term Cyclical Patterns (By Day and Hour)**:
Zooming in on the micro-trends reveals exactly when these issues occur.
- **By Day of Week**: There is a massive surge during the weekend. Saturdays and Sundays see nearly double the volume of a standard weekday (Monday through Thursday), with Friday serving as a transitional buildup day.
- **By Hour of Day**: The hourly distribution confirms that these are predominantly nighttime disturbances. Complaints peak heavily between 10:00 PM and Midnight, gradually tapering off to their lowest points in the early morning between 4:00 AM and 6:00 AM.

In [ ]:
# Plot 4: Hour-of-day and day-of-week
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['Created Date'].dt.hour.value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color='seagreen')
axes[0].set_title('Complaints by Hour of Day', fontweight='bold', fontsize=12)
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Count')

day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
df['Created Date'].dt.day_name().value_counts().reindex(day_order).plot(
    kind='bar', ax=axes[1], color='mediumpurple')
axes[1].set_title('Complaints by Day of Week', fontweight='bold', fontsize=12)
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

### Summary of Findings from Exploratory Data Analysis:
The exploratory analysis paints a clear picture: the complaints in this dataset are overwhelmingly residential, concentrated in the denser boroughs, and heavily biased toward typical "rest" periods (nights and weekends). Furthermore, the pandemic seems to have acted as a catalyst, permanently amplifying the seasonal volume of these complaints from 2020 onward.

## Data Analysis

### Did COVID change NYC's noise?

Did the pandemic leave a durable mark on the city's soundscape? We test it three ways..

1. How did monthly complaint volume change across the three eras?
2. Has the post-COVID level returned to baseline, or is the "new normal" higher?
3. Did the *mix* of complaint types shift, did residential noise rise while
   commercial/street noise fell?

Two first questions look at COVID's effect on the city's
soundscape. For those, we define three "eras":

- **Pre-COVID:** 2018-01-01 to 2020-02-29
- **Lockdown:** 2020-03-01 to 2021-06-30
- **Post-COVID:** 2021-07-01 to 2025-12-30

In [ ]:
# Label each complaint by COVID era for use in the analyses below
def covid_era(d):
    if d < pd.Timestamp('2020-03-01'):
        return 'Pre-COVID'
    elif d < pd.Timestamp('2021-07-01'):
        return 'Lockdown'
    else:
        return 'Post-COVID'

df['Era'] = df['Created Date'].map(covid_era)

#### The Volume Shift: A New Normal
As the time-series visualization illustrates, the onset of the lockdown triggered an immediate, massive spike in noise complaints. To quantify this, we compared the average monthly complaint volumes across our three eras:

In [ ]:
# 1 & 2: monthly complaint rate by era
monthly = df.set_index('Created Date').resample('MS').size()
era_monthly = df.groupby('Era').size() / df.groupby('Era')['Created Date'].apply(
    lambda s: (s.max() - s.min()).days / 30.44
)
era_monthly = era_monthly.round(0).astype(int).reindex(['Pre-COVID', 'Lockdown', 'Post-COVID'])

print("Average complaints per month by era:")
print(era_monthly.to_string())

pre = era_monthly['Pre-COVID']
lock = era_monthly['Lockdown']
post = era_monthly['Post-COVID']
print(f"\nLockdown vs pre:   {(lock/pre - 1)*100:+.1f}%")
print(f"Post-COVID vs pre: {(post/pre - 1)*100:+.1f}%")

# Plot the monthly series with era shading
fig, ax = plt.subplots(figsize=(13, 5))
monthly.plot(ax=ax, color='darkblue', linewidth=1.5)
ax.axvspan(pd.Timestamp('2020-03-01'), pd.Timestamp('2021-07-01'),
           alpha=0.15, color='red', label='Lockdown')
ax.set_title('Monthly noise complaints, 2018–2025', fontweight='bold')
ax.set_ylabel('Complaints per month')
ax.set_xlabel('')
ax.legend()
plt.tight_layout()
plt.show()

#### Findings (Q1 & Q2)
COVID's mark on NYC's noise is unmistakable and durable. During the lockdown, average monthly complaints nearly doubled (an 85% increase). Crucially, the city has not returned to its baseline. Five years later, the post-COVID average still runs roughly 68% above pre-pandemic levels.

#### The Composition Shift
While the total volume exploded, we also investigated if the source of the noise changed. To understand the immediate "shock" to the city's system, we specifically isolated the shift between the Pre-COVID and Lockdown eras. While the grouped bar chart tracks the full timeline through to our current post-pandemic reality, the horizontal bar chart zeroes in strictly on that initial lockdown period, highlighting the sheer magnitude of the shifts (in percentage points) when the city suddenly shut down.

In [ ]:
type_col = 'Complaint Type'
era_focus = df[df['Era'].isin(['Pre-COVID', 'Lockdown', 'Post-COVID'])]

share = (
    era_focus.groupby(['Era', type_col]).size()
    .groupby(level=0, group_keys=False)
    .apply(lambda s: s / s.sum() * 100)
    .unstack(fill_value=0)
 )

top_types = (
    share.loc['Pre-COVID'] + share.loc['Lockdown'] + share.loc['Post-COVID']
 ).sort_values(ascending=False).head(8).index
plot_df = share[top_types].T[['Pre-COVID', 'Lockdown', 'Post-COVID']]

fig, ax = plt.subplots(figsize=(10, 5))
plot_df.plot(kind='bar', ax=ax, color=['#4c78a8', '#f58518', '#54a24b'])
ax.set_title('Noise complaint type share: pre, lockdown, post', fontweight='bold')
ax.set_ylabel('Share of complaints (%)')
ax.set_xlabel('Complaint type')
ax.tick_params(axis='x', rotation=35)
ax.legend(title='Era')
plt.tight_layout()
plt.show()

In [ ]:
# 3: mix shift — top complaint types pre vs during lockdown
type_col = 'Complaint Type'
mix = (
    df[df['Era'].isin(['Pre-COVID', 'Lockdown'])]
    .groupby(['Era', type_col]).size()
    .unstack(fill_value=0)
    .apply(lambda r: r / r.sum() * 100, axis=1)
    .T
    .round(1)
 )
mix['Change (pp)'] = (mix['Lockdown'] - mix['Pre-COVID']).round(1)
mix = mix.sort_values('Change (pp)', ascending=False)
print("\nComplaint-type share (%) pre vs during lockdown:")
print(mix.to_string())

# Visualize the biggest shifts
top_change = mix['Change (pp)'].abs().sort_values(ascending=False).head(10)
fig, ax = plt.subplots(figsize=(8, 5))
top_change.sort_values().plot(kind='barh', ax=ax, color='slateblue')
ax.set_title('Largest complaint-type shifts (Lockdown vs Pre-COVID)', fontweight='bold')
ax.set_xlabel('Change in share (percentage points)')
ax.set_ylabel('Complaint type')
plt.tight_layout()
plt.show()

#### Findings (Q3)
Even as the total volume of complaints surged during the lockdown, the overall ranking of the categories did not change significantly. Because the lockdowns forced people to stay home, we initially expected a massive structural shift in the data—perhaps expecting Residential complaints to entirely dominate at the expense of all other types.

Instead, the baseline structure held its ground. Residential noise remained the undisputed primary issue, maintaining roughly half of the total share (shifting only slightly from 50.1% to 49.3%).

Rather than a total reshuffle of the data, we saw logical, specific shifts in the middle-tier categories that reflect the lockdown lifestyle:
- Taking it Outside: As socializing and dining were forced outdoors, Street/Sidewalk noise (+7.8 pp) and Vehicle noise (+2.4 pp) took up a slightly larger slice of the pie.
- The Business Shutdown: Predictably, Commercial noise shrank (-3.9 pp) as bars, clubs, and indoor venues shuttered.
- The Aerial Anomaly: Notably, Helicopter complaints more than tripled their share (+1.4 pp). The NYPD's heavy aerial surveillance of outdoor protests during the summer of 2020 is a highly plausible driver for this specific increase.

### When is NYC loudest?

Having established that COVID permanently altered the overall volume of noise complaints, we next wanted to understand the temporal rhythms of the city. The basic exploratory plots showed daily and hourly distributions separately, but the true story sits in their interaction (e.g., Friday at 2:00 AM is a very different environment than Monday at 2:00 AM).

To capture this, we generated a heatmap crossing the day of the week with the hour of the day. We then split that heatmap across our three COVID eras to see if the pandemic altered the fundamental weekly rhythm of the city. Finally, we looked at the broader seasonal seasonality across the entire dataset.

#### The Weekly Rhythm

In [ ]:
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

heatmap = (
    df.assign(
        dow=df['Created Date'].dt.day_name(),
        hour=df['Created Date'].dt.hour,
    )
    .pivot_table(index='dow', columns='hour', values='Borough',
                 aggfunc='count', fill_value=0)
    .reindex(day_order)
)

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(heatmap, cmap='YlOrRd', ax=ax, cbar_kws={'label': 'Complaints'})
ax.set_title('NYC noise complaints by day and hour', fontweight='bold')
ax.set_xlabel('Hour of day')
ax.set_ylabel('Day of week')
plt.tight_layout()
plt.show()

In [ ]:
# Day-of-week × hour, split by COVID era
fig, axes = plt.subplots(1, 3, figsize=(20, 5), sharey=True)
for ax, era in zip(axes, ['Pre-COVID', 'Lockdown', 'Post-COVID']):
    era_df = df[df['Era'] == era]
    heatmap = (
        era_df.assign(
            dow=era_df['Created Date'].dt.day_name(),
            hour=era_df['Created Date'].dt.hour,
        )
        .pivot_table(index='dow', columns='hour', values='Borough',
                     aggfunc='count', fill_value=0)
        .reindex(day_order)
    )
    # Normalize to complaints per day so the three eras are comparable
    n_days = (era_df['Created Date'].max() - era_df['Created Date'].min()).days
    heatmap = heatmap / n_days * 7  # avg per weekday slot

    sns.heatmap(heatmap, cmap='YlOrRd', ax=ax, cbar_kws={'label': 'Avg complaints'})
    ax.set_title(f'{era}', fontweight='bold')
    ax.set_xlabel('Hour of day')
    ax.set_ylabel('')

plt.suptitle('Day-of-week × hour, by COVID era', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Findings**

NYC has a distinct weekly rhythm characterized by a massive late-night weekend spike. While the shape of this rhythm persisted through COVID, its intensity permanently changed.

- The Weekend Ridge: The overall heatmap reveals a pronounced concentration of complaints late on Friday and Saturday nights. Specifically, the weekend hours between Midnight and 3:00 AM receive 3.25× the complaints compared to the equivalent weekday hours. Weekday hours remain comparatively flat, with only a mild evening uptick from roughly 8:00 PM to 11:00 PM.

- The Pandemic Effect: Splitting the data by era reveals that COVID amplified this rhythm but didn't structurally reshape it. All three eras show the exact same pattern: hottest on Saturday/Sunday from Midnight to 3:00 AM, and coolest during weekday mornings.

- The Intensity Shift: What changed was the sheer volume. The Pre-COVID era peaked at around ~290 complaints per time slot. During Lockdown, that peak surged to ~520 (roughly 1.8× higher). Crucially, Post-COVID still peaks around ~400 complaints. The city’s when didn't change; its how much did. This visually reinforces our earlier finding that the city never truly returned to its pre-pandemic baseline.

#### The Seasonal Rhythm
Beyond the weekly cycle, we examined the macro-level seasonality of noise complaints across the entire year.

In [ ]:
# Seasonality: complaints by month-of-year
monthly_avg = (
    df.groupby(df['Created Date'].dt.month).size() /
    df['Created Date'].dt.year.nunique()
).round(0).astype(int)
monthly_avg.index = ['Jan','Feb','Mar','Apr','May','Jun',
                     'Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(10, 4))
monthly_avg.plot(kind='bar', ax=ax, color='coral')
ax.set_title('Average monthly complaint volume by calendar month',
             fontweight='bold')
ax.set_ylabel('Complaints')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print(f"Seasonal ratio: peak month ({monthly_avg.idxmax()}) has "
      f"{monthly_avg.max() / monthly_avg.min():.2f}x the complaints of "
      f"quietest month ({monthly_avg.idxmin()}).")

# Quantify the weekend-night ridge
weekend_late = df[(df['Created Date'].dt.day_name().isin(['Saturday', 'Sunday'])) &
                  (df['Created Date'].dt.hour.isin([0, 1, 2, 3]))]
weekday_late = df[(df['Created Date'].dt.day_name().isin(['Tuesday', 'Wednesday', 'Thursday'])) &
                  (df['Created Date'].dt.hour.isin([0, 1, 2, 3]))]
ratio = (len(weekend_late) / 2) / (len(weekday_late) / 3)  # per-day normalization
print(f"Weekend midnight-3am hours get {ratio:.2f}x the complaints of weekday midnight-3am hours.")

In [ ]:
# Monthly complaint-type totals
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
top_types = df['Complaint Type'].value_counts().head(5).index.tolist()
monthly_types = (
    df[df['Complaint Type'].isin(top_types)]
    .groupby([df['Created Date'].dt.month, 'Complaint Type'])
    .size()
    .unstack(fill_value=0)
)
monthly_types = monthly_types.reindex(range(1, 13), fill_value=0)
monthly_types.index = month_order
monthly_types = (monthly_types / df['Created Date'].dt.year.nunique()).round(0).astype(int)

fig, ax = plt.subplots(figsize=(12, 6))
monthly_types.plot(ax=ax, linewidth=2)
ax.set_title('Monthly complaint-type totals', fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Complaints')
ax.legend(title='Complaint type')
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
# Seasonal share by COVID era
season_map = {1: 'Winter', 2: 'Winter', 12: 'Winter',
              3: 'Spring', 4: 'Spring', 5: 'Spring',
              6: 'Summer', 7: 'Summer', 8: 'Summer',
              9: 'Fall', 10: 'Fall', 11: 'Fall'}
season_order = ['Winter', 'Spring', 'Summer', 'Fall']
era_order = ['Pre-COVID', 'Lockdown', 'Post-COVID']

season_era = df[df['Era'].isin(era_order)].copy()
season_era['Season'] = season_era['Created Date'].dt.month.map(season_map)

share = (
    season_era.groupby(['Era', 'Season']).size()
    .groupby(level=0, group_keys=False)
    .apply(lambda s: s / s.sum() * 100)
    .unstack(fill_value=0)
)
share = share.reindex(index=era_order, columns=season_order)

fig, ax = plt.subplots(figsize=(10, 5))
share.T.plot(kind='bar', ax=ax, color=['#4c78a8', '#f58518', '#54a24b'])
ax.set_title('Season share by COVID era', fontweight='bold')
ax.set_ylabel('Share of complaints (%)')
ax.set_xlabel('Season')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Era')
plt.tight_layout()
plt.show()


**Findings**: The data reveals a dramatic seasonal curve that is clearly driven by warm weather, but a deeper look at the complaint types and eras reveals exactly why this happens.

- The Summer Peak: June is the peak month for noise complaints, registering 2.07× the volume of February, the quietest month.

- The Street Surge: The Monthly complaint-type totals line chart reveals that while Residential noise remains consistently high year-round, the summer spike is driven almost entirely by Street/Sidewalk noise. This perfectly aligns with urban summer living: more outdoor socializing combined with open windows makes street noise drastically more intrusive.

- The COVID Distortion: Finally, the Season share by COVID era chart reveals that the pandemic temporarily exaggerated this natural rhythm. During the Lockdown era, an unusually high share of the period's complaints were concentrated in the Spring and Summer months (over 65% combined), while the Winter share plummeted. This distinctly captures the unique reality of 2020, when the initial spring lockdowns gave way to a summer where New Yorkers socialized almost exclusively outdoors.

### Which boroughs are noisiest and with what kind of noise?

Raw complaint counts can be highly misleading because the boroughs differ enormously in size. For instance, Brooklyn has roughly 5.5× the population of Staten Island. If both were equally "loud" per person, we would expect Brooklyn to have 5.5× the complaints simply by virtue of its size.

To get a true sense of the city's noise landscape, we must normalize the data to look at per-capita complaint rates, and then break down the complaint-type composition of each individual borough.

#### Per-Capita Rates
By dividing the total complaints by the population of each borough, we get a much more accurate picture of the typical resident's experience:

In [ ]:
# 2020 US Census borough populations, used for per-capita rates in Q2
borough_pop = {
    'BRONX':         1_472_654,
    'BROOKLYN':      2_736_074,
    'MANHATTAN':     1_694_251,
    'QUEENS':        2_405_464,
    'STATEN ISLAND':   495_747,
}

In [ ]:
# Per-capita complaint rate by borough
borough_counts = df['Borough'].value_counts()
borough_df = pd.DataFrame({
    'Complaints': borough_counts,
    'Population': [borough_pop[b] for b in borough_counts.index],
})
borough_df['Per 1,000 residents'] = (
    borough_df['Complaints'] / borough_df['Population'] * 1000
).round(0).astype(int)
borough_df = borough_df.sort_values('Per 1,000 residents', ascending=False)
print("Noise complaints per 1,000 residents (2018–2025 total):")
print(borough_df.to_string())

# Plot: per-capita bar chart
fig, ax = plt.subplots(figsize=(10, 5))
borough_df['Per 1,000 residents'].plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Noise complaints per 1,000 residents, 2018–2025',
             fontweight='bold')
ax.set_ylabel('Complaints per 1,000 residents')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


**Finding: Raw totals and per-capita rates tell very different stories.**

The Bronx leads in both raw count (1.52M) and per-capita rate (1,031 complaints per 1,000 residents). That means, on average, every Bronx resident "generates" roughly one complaint per year. Manhattan is second (849 per 1,000), despite being only the third-largest borough by population.

The real surprise is Brooklyn. It is NYC's most populous borough, but per capita, it drops to fourth place at 497 (less than half the Bronx's rate). Staten Island is by far the quietest (242 per 1,000), experiencing roughly a quarter of the Bronx's complaint rate.

**The Timeline of Noise: The Bronx's Summer Peaks**

While the overall per-capita rates show the Bronx as the undisputed loudest borough over the 8-year period, mapping these rates over time reveals a much more nuanced story.


In [ ]:
# Monthly per-capita complaint trend by borough
borough_time_df = (
    df.assign(Month=df['Created Date'].dt.to_period('M').dt.to_timestamp())
    .groupby(['Month', 'Borough']).size()
    .reset_index(name='Complaints')
)
borough_time_df['Population'] = borough_time_df['Borough'].map(borough_pop)
borough_time_df['Per 1,000 residents'] = (
    borough_time_df['Complaints'] / borough_time_df['Population'] * 1000
).round(2)
borough_time_pivot = borough_time_df.pivot(index='Month', columns='Borough', values='Per 1,000 residents')
borough_time_pivot = borough_time_pivot.reindex(columns=['BRONX', 'BROOKLYN', 'MANHATTAN', 'QUEENS', 'STATEN ISLAND'])

fig, ax = plt.subplots(figsize=(12, 6))
borough_time_pivot.plot(ax=ax, linewidth=2)
ax.set_title('Monthly noise complaints per 1,000 residents by borough, 2018–2025', fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Complaints per 1,000 residents')
ax.legend(title='Borough')
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()


**Finding: The Bronx's lead is driven by explosive seasonal spikes.** Looking at the timeline, the hierarchy of noise is not static:
- **The Pre-COVID Reality:** Before 2020, Manhattan (the green line) was frequently the loudest borough per capita, occasionally trading the number one spot with the Bronx depending on the season.
- **The Post-2020 Bronx Shift:** The pandemic completely fractured this pattern. Starting in the summer of 2020, the Bronx began experiencing massive, disproportionate summer spikes. While all boroughs show a warm-weather increase, the Bronx's summer peaks are explosive—often reaching over 30 complaints per 1,000 residents in a single month, far exceeding the seasonal variations of the other four boroughs.
- **The Quiet Baseline:** Meanwhile, Brooklyn, Queens, and Staten Island remain tightly grouped at the bottom of the chart, following a much gentler, highly predictable seasonal wave year after year.

#### Borough Profiles

Beyond the volume of noise, we wanted to know if the character of the noise changed depending on where you are in the city. We first looked at the broad categories.

In [ ]:
# Complaint-type composition per borough — top 3 types per borough
type_col = 'Complaint Type'
by_bor = (
    df.groupby(['Borough', type_col]).size()
    .groupby(level=0, group_keys=False)
    .apply(lambda s: (s / s.sum() * 100).round(1).nlargest(3))
)
print("\nTop 3 complaint types per borough (% share):")
print(by_bor.to_string())

**Findings:** Raw totals and per-capita rates tell very different stories.

The **Bronx leads in both raw count (1.52M) and per-capita rate (1,031 complaints per 1,000 residents)**. That means every Bronx resident "generates" roughly one complaint per year on average. **Manhattan is second (849 per 1,000)**, despite being only the third-largest borough by population. The real surprise is Brooklyn: it's NYC's most populous borough, but per capita it ranks fourth at 497, which is less than half the Bronx's rate. **Staten Island is by far the quietest (242 per 1,000)**, roughly a quarter of the Bronx's rate.

The *composition* also differs sharply:
- **Bronx** is the most residential-heavy (63.7% residential). Complaints are overwhelmingly about neighbors.
- **Manhattan** has the most diverse mix, with residential at only 33.8% and street/sidewalk noise a very close second at 28.3%, consistent with its commercial, tourist-heavy character.
- **Queens and Staten Island** look structurally similar to the Bronx, i.e. residential-dominant.

This tells us borough-level maps need to be **normalized by population** to avoid simply showing a map of where people live, and that "loud" means genuinely different things in different boroughs. The finding drives **The Main Event** (interactive choropleth map), and the type-mix numbers feed **The Neighborhood Showdown** (radar charts comparing borough profiles).

**The Borough Fingerprint (By Descriptor):**


In [ ]:
# The Borough Fingerprint: top 5 descriptors per borough
desc_col = 'Complaint description'

filtered = df[df[desc_col].notna()]
top_desc_by_bor = (
    filtered.groupby(['Borough', desc_col]).size()
    .groupby(level=0, group_keys=False)
    .apply(lambda s: s.nlargest(5))
    .reset_index(name='Count')
 )

# Build a consistent descriptor set across boroughs for the heatmap
descriptor_set = sorted(top_desc_by_bor[desc_col].unique())
heatmap_df = (
    filtered[filtered[desc_col].isin(descriptor_set)]
    .groupby(['Borough', desc_col]).size()
    .unstack(fill_value=0)
 )

# Convert to share (%) within borough to make boroughs comparable
heatmap_share = (
    heatmap_df.T / heatmap_df.T.sum(axis=0) * 100
 ).round(2)

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(heatmap_share, cmap='YlGnBu', ax=ax, cbar_kws={'label': 'Share of borough complaints (%)'})
ax.set_title('Top descriptors by borough', fontweight='bold')
ax.set_xlabel('Borough')
ax.set_ylabel('Descriptor')
plt.tight_layout()
plt.show()

To get even more specific we drilled down past the broad categories into the specific descriptors. By visualizing these descriptors as a percentage of each borough's total noise, distinct "fingerprints" emerge:

- The Universal Soundtrack: The most striking takeaway from the heatmap is universality. "Loud Music/Party" (the dark blue band) is the undisputed king of noise across the board, dominating the complaint share in all five boroughs, though it peaks at its absolute highest intensity in the Bronx.

- The Residential Echo: "Banging/Pounding" represents the city's clear secondary acoustic baseline. However, its intensity is strongest in the Bronx, Brooklyn, and Queens, highlighting the friction of high-density apartment living in the outer boroughs.

- Manhattan's Development Hum: Looking at the lighter bands, Manhattan shows uniquely visible activity in "Noise: Construction Before/After Hours (NM1)" and the general "Other" category compared to the rest of the city, perfectly reflecting its status as a dense, constantly developing commercial hub.

- Staten Island's Suburbs: Conversely, Staten Island shows a slightly higher relative share in the "Noise, Barking Dog (NR5)" row compared to boroughs like Manhattan, reflecting its less dense, more suburban layout with single-family homes and yards.

### Where are the hotspots?

While borough-level metrics provide a good baseline, a borough is simply too coarse a geographic unit to tell the whole story. Brooklyn alone is home to 2.7 million people and encompasses neighborhoods as wildly different as Williamsburg and Bay Ridge.

To find the true sources of the noise, we needed to get much more granular. We aggregated the complaints into a fine geographic coordinate grid (rounding to 3 decimal places, which creates roughly a 100-meter by 100-meter cell) to identify the city's absolute loudest micro-locations.

In [ ]:
# Round coordinates to ~100m grid and count complaints per cell
grid = (
    df.assign(
        lat_grid=df['Latitude'].round(3),
        lng_grid=df['Longitude'].round(3),
    )
    .groupby(['lat_grid', 'lng_grid', 'Borough'])
    .size()
    .reset_index(name='Complaints')
    .sort_values('Complaints', ascending=False)
)
print(f"{len(grid):,} distinct grid cells contain complaints.")
print(f"Top 15 hotspots over the full period:\n")
print(grid.head(15).to_string(index=False))

# How concentrated is the noise? What share of complaints comes from the top 1% of cells?
total = grid['Complaints'].sum()
top_1pct_cutoff = int(len(grid) * 0.01)
top_1pct_share = grid.head(top_1pct_cutoff)['Complaints'].sum() / total * 100
print(f"\nTop 1% of cells ({top_1pct_cutoff:,} cells) account for "
      f"{top_1pct_share:.1f}% of all complaints.")

**Finding:** NYC's noise is extraordinarily concentrated. The top 1% of ~100m grid cells (just 555 cells out of 55,543) account for **27.5% of all complaints**.

However, the top of the list comes with a strong caveat. The single loudest grid cell (40.892, -73.860 in the Bronx) has **249,553 complaints**, and the second cell (40.892, -73.859), essentially the same location, ~85m away, adds another 142,481. Together, these two adjacent cells produce 7.2% of the entire dataset. **This is almost certainly not a real noise hotspot but a coordinate artifact**. Either it's a default coordinate assigned by 311 when an address can't be geocoded, or a single large residential complex generating disproportionate reports.

Setting that anomaly aside, the real hotspot pattern is instructive: the top cells are split across all four major boroughs, showing noise concentration isn't a single-borough phenomenon.

# Interactive Maps

Geojson data is from https://data.cityofnewyork.us/City-Government/Borough-Boundaries/gthc-hcne/about_data

NOTE: I had to outcomment the code for both maps cause the file is too large and cannot fit to git limits of 100 MB. 

### Borough Comparison Interactive map


The first interactive tool enables the user to pick a complaint type (from all of the complain types) and an era, so pre-Covid, lockdown and post-covid and see how the noice complaints change. This is however aggregated over boroughs, for higher overview. We will later zoom in on neighborhoods.  

In [ ]:
import json
import pandas as pd
import plotly.express as px

agg = (
    df.groupby(['Borough', 'Complaint description', 'Era'])
    .size()
    .reset_index(name='Count')
)
#map borough names to match geojson
boroname_map = {'MANHATTAN': 'Manhattan', 'BRONX': 'Bronx', 'BROOKLYN': 'Brooklyn',
                'QUEENS': 'Queens', 'STATEN ISLAND': 'Staten Island'}
agg['Borough_geo'] = agg['Borough'].map(boroname_map)

with open('C:/git/sda_final_project/notebook/Borough_Boundaries_20260429.geojson', 'r') as f:
    geojson = json.load(f)

# Keep only complaint types that have data in all 3 eras (prevents blank maps)
era_coverage = (
    agg.groupby(['Complaint description', 'Era'])['Count'].sum()
    .unstack(fill_value=0)
)
valid_descs = sorted(
    era_coverage[era_coverage[['Pre-COVID', 'Lockdown', 'Post-COVID']].gt(0).all(axis=1)].index
)

agg = agg[agg['Complaint description'].isin(valid_descs)]

all_boroughs = pd.DataFrame({'Borough_geo': ['Manhattan', 'Bronx', 'Brooklyn', 'Queens', 'Staten Island']})

def get_data(complaint_desc, complaint_era):
    subset = agg[(agg['Complaint description'] == complaint_desc) & (agg['Era'] == complaint_era)]
    # Always return all 5 boroughs — missing ones get 0 instead of blank map
    return all_boroughs.merge(subset[['Borough_geo', 'Count']], on='Borough_geo', how='left').fillna(0)

desc = 'Loud Music/Party'
era  = 'Pre-COVID'
eras = ['Pre-COVID', 'Lockdown', 'Post-COVID']

filtered = get_data(desc, era)

# fig = px.choropleth(
#     filtered,
#     geojson=geojson,
#     locations='Borough_geo',
#     featureidkey='properties.boroname',
#     color='Count',
#     color_continuous_scale='YlOrRd',
#     labels={'Count': 'Complaints'},
# )
# fig.update_geos(fitbounds="locations", visible=False)

# buttons_desc = []
# for d in valid_descs:
#     df_d = get_data(d, era)
#     buttons_desc.append(dict(
#         label=d, method='update',
#         args=[{'z': [df_d['Count'].tolist()], 'locations': [df_d['Borough_geo'].tolist()]},
#               {'title.text': f'NYC Noise Complaints: {d} ({era})'}]))

# buttons_era = []
# for e in eras:
#     df_e = get_data(desc, e)
#     buttons_era.append(dict(
#         label=e, method='update',
#         args=[{'z': [df_e['Count'].tolist()], 'locations': [df_e['Borough_geo'].tolist()]},
#               {'title.text': f'NYC Noise Complaints: {desc} ({e})'}]))

# fig.update_layout(
#     title=dict(
#         text=f'NYC Noise Complaints: {desc} ({era})',
#         x=0.5, xanchor='center',
#         y=0.97, yanchor='top',
#         font=dict(size=18)
#     ),
#     updatemenus=[
#         dict(buttons=buttons_desc, direction='down', showactive=True,
#              x=0.05, xanchor='left', y=1.12, yanchor='top', bgcolor='white'),
#         dict(buttons=buttons_era,  direction='down', showactive=True,
#              x=0.55, xanchor='left', y=1.12, yanchor='top', bgcolor='white'),
#     ],
#     annotations=[
#         dict(text='<b>Complaint type:</b>', x=0.05, y=1.12, yanchor='bottom',
#              xref='paper', yref='paper', showarrow=False, align='left'),
#         dict(text='<b>Era:</b>', x=0.55, y=1.12, yanchor='bottom',
#              xref='paper', yref='paper', showarrow=False, align='left'),
#     ],
#     autosize=True,
#     margin={"r": 0, "t": 120, "l": 0, "b": 0}
# )

### Neighborhooods Over Time Interactive Map
GeoJson from: https://github.com/veltman/snd3/blame/master/data/nyc-neighborhoods.geo.json

An interactive tool where the user can see all neighborhoods and see their noise profiles compared side-by-side. There is also a year slider, so you can watch the neighborhoods change over time. The top 3 most common complaints have been chosen for simplicity. When you hover over the area, you also see the most common complaint at a given time. 


In [ ]:
import pandas as pd
import plotly.express as px
import json
from shapely.geometry import Point, shape
from shapely.strtree import STRtree
import numpy as np

with open('NYC_Neighborhoods.geojson', 'r') as f:
    nta_geojson = json.load(f)

nta_geoms     = [shape(f['geometry']) for f in nta_geojson['features']]
nta_names_list = [f['properties']['NTAName'] for f in nta_geojson['features']]
tree = STRtree(nta_geoms)

# def find_neighborhood(lat, lon):
#     point = Point(lon, lat)
#     idx = tree.nearest(point)
#     if isinstance(idx, (int, np.integer)):
#         return nta_names_list[idx] if nta_geoms[idx].contains(point) else None
#     return None

# df['Year'] = df['Created Date'].dt.year

# top_noise_types = df['Complaint Type'].value_counts().nlargest(5).index.tolist()
# df_filtered = df[df['Complaint Type'].isin(top_noise_types)].copy()
# df_filtered['Neighborhood'] = df_filtered.apply(
#     lambda r: find_neighborhood(r['Latitude'], r['Longitude']), axis=1
# )
# df_mapped = df_filtered.dropna(subset=['Neighborhood']).copy()

# agg = (
#     df_mapped.groupby(['Neighborhood', 'Year', 'Complaint Type'])
#     .size()
#     .reset_index(name='Count')
# )

# # Most frequent noise type per neighborhood + year
# top_type = (
#     agg.loc[agg.groupby(['Neighborhood', 'Year'])['Count'].idxmax(),
#             ['Neighborhood', 'Year', 'Complaint Type']]
#     .rename(columns={'Complaint Type': 'Top Type'})
# )

# # One row per (Neighborhood, Year): total count + top type for hover
# agg_total = (
#     agg.groupby(['Neighborhood', 'Year'])['Count']
#     .sum()
#     .reset_index()
#     .merge(top_type, on=['Neighborhood', 'Year'])
# )

# HOVER = (
#     '<b>%{hovertext}</b><br>'
#     'Complaints: %{customdata[1]:.0f}<br>'
#     'Most frequent type: %{customdata[0]}'
#     '<extra></extra>'
# )

# fig = px.choropleth_mapbox(
#     agg_total,
#     geojson=nta_geojson,
#     locations='Neighborhood',
#     featureidkey="properties.NTAName",
#     color='Count',
#     animation_frame='Year',
#     hover_name='Neighborhood',
#     custom_data=['Top Type', 'Count'],
#     color_continuous_scale="Viridis",
#     range_color=[0, agg_total['Count'].quantile(0.95)],
#     mapbox_style="carto-positron",
#     zoom=9.5,
#     center={"lat": 40.7128, "lon": -74.0060},
#     opacity=0.7,
#     labels={'Count': 'Complaints'},
#     title='NYC Noise Complaints by Neighborhood'
# )

# Base trace + every animation frame need the template set explicitly
# fig.update_traces(hovertemplate=HOVER)
# for frame in fig.frames:
#     frame.data[0].update(hovertemplate=HOVER)

# unique_types = agg['Complaint Type'].unique()
# buttons = [
#     dict(method="restyle", label=n_type,
#          args=[{"z": [agg[agg['Complaint Type'] == n_type]['Count']]}])
#     for n_type in unique_types
# ]

# fig.update_layout(
#     updatemenus=[dict(buttons=buttons, direction="down",
#                       showactive=True, x=0.1, y=1.15)],
#     margin={"r": 0, "t": 40, "l": 0, "b": 0}
# )



## Genre and Narratives choices

For overall Genre, we have decided to go with Magazine Style article. We felt like we had a story to tell and wanted to guide users through the reading, instead of making a website where you need to navigate yourself and it might be confusing. We wanted to lead the reader through our analysis.

### Use of Visual Narratives
For each new research question, we have used 'highlighting' and phrase it as a question to draw users attention, so the headlines were in bold. From 'Visual Structuring' we have tried to make the website a consistent visual platform. The linear scroll of a website works quite well for this. We have also ensured that there is some sort of text description in between the research question where we are trying to guide the user through, which could be characterized as 'Transition Guidance'. This is to make sure that we keep the user engaged and that the story gets through.  

### Use of Narrative Structures

From Narrative structure we have used 'Ordering', specifically linear, since our website naturally guides the user through the story. To many of the charts we have added 'Interactivity' and also developed 2 extra interactive tools. This is because it feels natural when you talk about a city, that you want a user to be able to interact with a map and explore the areas they are specifically interested it. It also makes them more engaged, as they can zoom in on their apartment or on places they are interested in. For 'Messaging'
Each figure has caption and a description of what we want the user to notice, but of course for for example the interactive charts, the user might also explore it by their own interest. The captions should just help reader to understand before they dive into the graph or a tool. We have also made sure to add introduction and a summary, to ensure that after all the interactive tools, the user gets what was our intention behind this reasearch and what we think should be the key takeaway. 




## Visualizations.

Explain the visualizations you've chosen. Why are they right for the story you want to tell?

In our project we have decided to focus on wide variaty of visualization to utilize as much of the course content in this project, of course we also tried to make sure that this is right tool for the job. We started with 'Line chart', showing the overall volume of complaints over time in New York to introduce the story. 
We then used a 'Bar Chart' to show how the complaint types change over time, especially if you compare it pre, during and post covid. The bar chart seemed like a good tool for such comparison. To example the distribution patterns, the natural choice was the usage of a heatmaps, as they are an excellent tool for pattern visualization. We then followed by another bar cahrt, here we wanted to show that the noice complaints per borough but we wanted to make sure they are of course adjusted for population size. Bar chart is a tool that enables communication in a simple way when it comes to this topic. Next visualization is a multi line chart, which is interactive, here we wanted to show how the trends are changing over time, adjusted per pupulation size. This is because we stumpled upon a previous research, that was done pre covid, and Bronx was not as 'noisy' back then, so we wanted to show that this and Manhattan area seem to have a pattern of spikes. To examine the seasonal patterns, we have again used a bar chart, where a user can compare the seasonal shirt in complaints pre, during and post covid era. 

Moving to the last 2 interactive tools, here in both cases we have selected an interactive map, but with a different level of granularity and focus. The first map focuses on boroughs, as we wanted to show the overall trends and the changes in for example Collection Truck Noise, pre, during and post pandemic. This is to catch users interest in this topic, as you can clearly see that the overall trend has shifted. For the second visualization, we wanted to dig deeper into neighborhoods, so we have chosen an interactive map, but this time zoomed into all neighborhoods over time. To limit the complexity, here we only allow users to see the top 3 most common complain types. There is a play button and a slider where a user can see how the trend changes over time in realtime. We have chosen this to hopefully attrack the user even more, as they might try to see how their neighborhood changed, or whether to move into a certain area. 




## Discussion. Think critically about your creation

What went well?

The data was fairly structured and the analysis revealed some interesting patterns, like this COVID change which made us more engaged with the story. 
The website looks simple, but that is intentional as we care more about the story we want to tell, we wanted it to feel like an investigation. 

What is still missing? What could be improved? Why?

As it was said in the podcast we had for this course, the best analysis is done when you fully know the domain. As we are all from outside of US, we might be missing some explanations or reasoning why we see some behavioral and complain types shift. 
We also do not really know if they made any update to this system, and whether it is now simply easier to put a complain, which might also explain the increase, or whether this is a cultural shift, where people just like to complain more. 

It would be interesting to merge this data with some other social dat (like weather data), but we decided to leave this stuff out due to the time constraint and because we actually had a story without having to add more data to it which would have made it measier. 


## Contributions. Who did what?

Names and Student numbers: Veronika Lietavcova (250695), Julia Papiernik (254743), Thorri Elis Halldoruson (s252362)
- Data Gathering and Cleaning: Thorri
- Data Analysis, Descriptions: Julia
- Data Visualization Non Interactive: Julia
- Interactive Data Visualizations: Veronika
- Storytelling and Website structure: Veronika
- Final Notebook refactoring and organizing: Veronika and Julia 
